# 01.2 — Distribution Shape & Intermittency Analysis

## 1. Mục tiêu & Ranh giới Kiến trúc
Notebook này thực hiện hai phân tích có ranh giới phương pháp luận độc lập:

```text
01.2 Distribution & Intermittency Analysis
│
├── A. Distribution Analysis (Phân tích phân phối lượng mưa)
│      └── Khảo sát biến liên tục Y_t >= 0 (mm), HOÀN TOÀN ĐỘC LẬP VỚI THRESHOLD.
│          Bao gồm: Thống kê mô tả, phân rã zero mass vs positive rainfall,
│          hình thái lệch phải (skewness/kurtosis), và phép biến đổi log1p.
│
└── B. Intermittency Diagnostics (Chẩn đoán tính gián đoạn)
       └── Chẩn đoán phụ trợ (auxiliary diagnostic) khảo sát cấu trúc xuất hiện.
           Thống nhất một canonical threshold duy nhất cho toàn project: Y_t >= 1.0 mm (Wet).
           So sánh với candidate nhạy cảm tau = 0.1 mm (Y_t >= 0.1 mm).
           Áp dụng tương quan taxonomy Syntetos-Boylan (từ demand forecasting) như một phép loại suy định lượng.
```

### Nguyên tắc Bằng chứng Khoa học (Evidence Boundaries):
1. **Phân phối chính không phụ thuộc ngưỡng**: Thống kê của biến mục tiêu liên tục $Y_t$ (Mean, Median, Std, Skewness, Kurtosis, Percentiles) được tính trên toàn bộ dữ liệu thô $Y_t \ge 0$.
2. **Thống nhất Một Canonical Wet-Day Semantics ($Y_t \ge 1.0\text{ mm}$)**:
   Toàn bộ project thống nhất định nghĩa duy nhất cho wet-day occurrence:
   $$\boxed{\text{Wet Day: } Y_t \ge 1.0\text{ mm/day}} \quad \Longleftrightarrow \quad \text{Dry Day: } Y_t < 1.0\text{ mm/day}$$
   Áp dụng đồng bộ xuyên suốt từ EDA (01.1, 01.2) đến Hurdle Stage 1 Occurrence và secondary rain-day diagnostics, không chia tách hai convention khác nhau.
3. **Chuẩn hóa thuật ngữ & Boundary Censoring Robustness**:
   - **Wet days / Positive observations**: Tổng số ngày quan sát đạt ngưỡng ($Y_t \ge \tau$).
   - **Precipitation episodes**: Chuỗi ngày mưa liên tiếp xác định theo onset ($Y_t \ge \tau \land Y_{t-1} < \tau$).
   - **Boundary policy**: Thống kê đợt mưa được tính trong cửa sổ Dev Train (2001-01-01 → 2018-04-30, N = 6,329). Ngày đầu (2001-01-01 = 0.0 mm) không bị left-censored. Báo cáo song song cả `mean_duration_all` (toàn bộ đợt trong cửa sổ) và `mean_duration_uncensored` (loại trừ đợt mưa bị cắt tại biên phải nếu có) để bảo đảm tính vững (robustness).
4. **Bản chất của Syntetos-Boylan**: Taxonomy này bắt nguồn từ dự báo nhu cầu (demand forecasting), được sử dụng ở đây như một **công cụ chẩn đoán loại suy (exploratory diagnostic analogy)** để gợi ý họ mô hình (Two-Stage Hurdle / Tweedie), không phải phân loại khí tượng chính thức và không khẳng định mô hình nào chắc chắn tối ưu.
5. **Phân loại cường độ khí tượng**: Dẫn chiếu chính xác theo **QCVN 47:2018/BTNMT & Quyết định 18/2021/QĐ-TTg** của cơ quan khí tượng Việt Nam.
6. **EDA motivates, not proves**: EDA cung cấp bằng chứng để đề xuất các họ mô hình đáng đánh giá (*motivates evaluation*), không tuyên bố OLS thất bại hay khẳng định bất kỳ mô hình nào tối ưu trước thử nghiệm.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.evaluation.protocols import load_canonical_data
from src.config.constants import Config

# 1. Nạp tập Dev Train chuẩn (Zero Leakage: 2001-01-01 -> 2018-04-30, N = 6,329)
dev_train, val_set, test_set = load_canonical_data(return_dev_train=True)
train_df = dev_train
target = train_df['Lượng mưa'].dropna()
dates = pd.to_datetime(train_df['Ngày'])
n_total = len(target)

print(f"📊 Tập Dev Train phân tích: {n_total:,} ngày ({train_df['Ngày'].min().date()} → {train_df['Ngày'].max().date()})")

# 2. Thống kê Phân phối Toàn bộ Biến Mục tiêu Liên tục Y_t (Threshold-Independent)
mean_all = float(target.mean())
std_all = float(target.std())
median_all = float(target.median())
cv_all = (std_all / mean_all) * 100.0 if mean_all > 0 else 0.0
skew_all = float(target.skew())
kurt_all = float(target.kurt())
q1_all = float(target.quantile(0.25))
q3_all = float(target.quantile(0.75))
iqr_all = q3_all - q1_all
p95_all = float(target.quantile(0.95))
p99_all = float(target.quantile(0.99))
max_all = float(target.max())

print("\n📈 [PHẦN 1] THỐNG KÊ PHÂN PHỐI LƯỢNG MƯA TOÀN BỘ (Y_t >= 0 mm):")
print(f"   • Số quan sát:             {n_total:,} ngày")
print(f"   • Mean (Trung bình):       {mean_all:.2f} mm")
print(f"   • Median (Trung vị):       {median_all:.2f} mm (Mode = 0.0 mm)")
print(f"   • Std (Độ lệch chuẩn):     {std_all:.2f} mm")
print(f"   • Overall CV:              {cv_all:.2f}% (Bị ảnh hưởng đồng thời bởi zero mass và heavy tail)")
print(f"   • Skewness (Độ lệch):      {skew_all:.2f} (Lệch phải mạnh)")
print(f"   • Excess Kurtosis:         {kurt_all:.2f} (Đuôi rất nặng - Leptokurtic)")
print(f"   • Q1 - Q3 (IQR):           [{q1_all:.2f}, {q3_all:.2f}] mm (IQR = {iqr_all:.2f} mm)")
print(f"   • P95 - P99:               P95 = {p95_all:.2f} mm, P99 = {p99_all:.2f} mm, Max = {max_all:.2f} mm")

# 3. Phân rã Zero Mass vs Positive Rainfall
zero_mask = (target == 0.0)
pos_mask = (target > 0.0)
n_zero = int(zero_mask.sum())
n_pos = int(pos_mask.sum())
pct_zero = (n_zero / n_total) * 100.0
pct_pos = (n_pos / n_total) * 100.0

target_pos = target[pos_mask]
pos_mean = float(target_pos.mean())
pos_std = float(target_pos.std())
pos_median = float(target_pos.median())
pos_cv = (pos_std / pos_mean) * 100.0 if pos_mean > 0 else 0.0
pos_q1 = float(target_pos.quantile(0.25))
pos_q3 = float(target_pos.quantile(0.75))
pos_p95 = float(target_pos.quantile(0.95))
pos_p99 = float(target_pos.quantile(0.99))

print("\n🔬 [PHẦN 2] PHÂN RÃ CẤU TRÚC: ZERO MASS VS POSITIVE RAINFALL:")
print(f"   • Exact-Zero Mass (Y = 0):  {n_zero:,} ngày ({pct_zero:.2f}% tổng dữ liệu)")
print(f"   • Positive Rainfall (Y > 0): {n_pos:,} ngày ({pct_pos:.2f}% tổng dữ liệu)")
print(f"     - Mean (lượng mưa có điều kiện Y > 0): {pos_mean:.2f} mm")
print(f"     - Median (Y > 0):                      {pos_median:.2f} mm (Q1 = {pos_q1:.2f}, Q3 = {pos_q3:.2f})")
print(f"     - Positive Population CV:              {pos_cv:.2f}%")
print(f"     - Cực trị dương (P95, P99, Max):       P95 = {pos_p95:.2f} mm, P99 = {pos_p99:.2f} mm, Max = {max_all:.2f} mm")
print(f"   💡 Nhận xét: CV giảm từ {cv_all:.2f}% (toàn bộ) xuống {pos_cv:.2f}% (khi lọc zero mass),")
print("      cho thấy sự kết hợp giữa điểm mass tại 0 và độ biến thiên lớn của các ngày có mưa.")


📊 Tập Dev Train phân tích: 6,329 ngày (2001-01-01 → 2018-04-30)

📈 [PHẦN 1] THỐNG KÊ PHÂN PHỐI LƯỢNG MƯA TOÀN BỘ (Y_t >= 0 mm):
   • Số quan sát:             6,329 ngày
   • Mean (Trung bình):       3.59 mm
   • Median (Trung vị):       0.50 mm (Mode = 0.0 mm)
   • Std (Độ lệch chuẩn):     7.14 mm
   • Overall CV:              198.73% (Bị ảnh hưởng đồng thời bởi zero mass và heavy tail)
   • Skewness (Độ lệch):      4.02 (Lệch phải mạnh)
   • Excess Kurtosis:         26.03 (Đuôi rất nặng - Leptokurtic)
   • Q1 - Q3 (IQR):           [0.00, 3.89] mm (IQR = 3.89 mm)
   • P95 - P99:               P95 = 17.37 mm, P99 = 34.18 mm, Max = 107.90 mm

🔬 [PHẦN 2] PHÂN RÃ CẤU TRÚC: ZERO MASS VS POSITIVE RAINFALL:
   • Exact-Zero Mass (Y = 0):  1,729 ngày (27.32% tổng dữ liệu)
   • Positive Rainfall (Y > 0): 4,600 ngày (72.68% tổng dữ liệu)
     - Mean (lượng mưa có điều kiện Y > 0): 4.94 mm
     - Median (Y > 0):                      1.77 mm (Q1 = 0.27, Q3 = 6.32)
     - Positive Population CV:    

In [2]:
# 4. Chẩn đoán Phép Biến đổi Hình thái: Raw Scale vs Log1p Scale
target_log1p = np.log1p(target)
skew_log1p = float(target_log1p.skew())
kurt_log1p = float(target_log1p.kurt())

print("\n📐 [PHẦN 3] CHẨN ĐOÁN HÌNH THÁI VÀ PHÉP BIẾN ĐỔI LOG1P:")
print(f"   • Thang đo thô Y:        Skewness = {skew_all:.2f}, Kurtosis = {kurt_all:.2f} (Lệch phải mạnh)")
print(f"   • Thang đo log(1 + Y):   Skewness = {skew_log1p:.2f}, Kurtosis = {kurt_log1p:.2f} (Đối xứng hơn rõ rệt)")
print("   💡 Ý nghĩa thực nghiệm: Phép biến đổi log1p làm giảm độ lệch và độ nhọn đuôi rõ rệt,")
print("      cho thấy không gian biểu diễn log-transformed là một ứng viên đáng đánh giá thử nghiệm trong Modeling.")
print("      Lưu ý: Mô hình train trên log1p(Y) phải inverse-transform (expm1) trước khi tính các metric MAE/RMSE/CR-MAE thô.")

# 5. Phân loại Cường độ Lượng mưa theo Quy chuẩn Khí tượng Việt Nam
# Dẫn chiếu: QCVN 47:2018/BTNMT & Quyết định số 18/2021/QĐ-TTg của Thủ tướng Chính phủ
categories_def = Config.PRECIPITATION_CLASSIFICATION["categories"]
vn_records = []

for cat_key, cat_info in categories_def.items():
    min_val, max_val = cat_info["range"]
    if cat_key == "no_rain":
        cnt = int((target == 0.0).sum())
        cond_str = "Y == 0 mm"
    elif max_val == float('inf'):
        cnt = int((target > min_val).sum())
        cond_str = f"Y > {min_val:.1f} mm"
    else:
        cnt = int(((target > min_val) & (target <= max_val)).sum())
        cond_str = f"{min_val:.1f} < Y <= {max_val:.1f} mm"
    
    pct = (cnt / n_total) * 100.0
    vn_records.append({
        "Mã nhóm": cat_key,
        "Phân loại (Tiếng Việt)": cat_info["label_vi"],
        "Khoảng ngưỡng 24h": cond_str,
        "Số ngày": cnt,
        "Tỷ lệ (%)": round(pct, 2)
    })

df_vn_classes = pd.DataFrame(vn_records)
print("\n🌧️ [PHẦN 4] PHÂN BỐ CƯỜNG ĐỘ THEO QUY CHUẨN KHÍ TƯỢNG VIỆT NAM (QCVN 47:2018/BTNMT):")
print(df_vn_classes.to_string(index=False))



📐 [PHẦN 3] CHẨN ĐOÁN HÌNH THÁI VÀ PHÉP BIẾN ĐỔI LOG1P:
   • Thang đo thô Y:        Skewness = 4.02, Kurtosis = 26.03 (Lệch phải mạnh)
   • Thang đo log(1 + Y):   Skewness = 0.98, Kurtosis = -0.18 (Đối xứng hơn rõ rệt)
   💡 Ý nghĩa thực nghiệm: Phép biến đổi log1p làm giảm độ lệch và độ nhọn đuôi rõ rệt,
      cho thấy không gian biểu diễn log-transformed là một ứng viên đáng đánh giá thử nghiệm trong Modeling.
      Lưu ý: Mô hình train trên log1p(Y) phải inverse-transform (expm1) trước khi tính các metric MAE/RMSE/CR-MAE thô.

🌧️ [PHẦN 4] PHÂN BỐ CƯỜNG ĐỘ THEO QUY CHUẨN KHÍ TƯỢNG VIỆT NAM (QCVN 47:2018/BTNMT):
             Mã nhóm  Phân loại (Tiếng Việt)    Khoảng ngưỡng 24h  Số ngày  Tỷ lệ (%)
             no_rain               Không mưa            Y == 0 mm     1729      27.32
          trace_rain Mưa lượng không đáng kể    0.0 < Y <= 0.6 mm     1530      24.17
          light_rain                 Mưa nhỏ    0.6 < Y <= 6.0 mm     1885      29.78
       moderate_rain                

In [3]:
# 6. Chẩn đoán Tính Gián đoạn (Intermittency Diagnostics - Auxiliary)
# Thống nhất một canonical wet-day definition duy nhất xuyên suốt project: Y >= 1.0 mm (ETCCDI RR1 standard)
# So sánh với candidate nhạy cảm: tau = 0.1 mm (Y >= 0.1 mm)
interm_records = []

eval_configs = [
    {"th": 1.0, "status": "Canonical Project Definition", "desc": "tau=1.0 mm (Y >= 1.0 mm, Canonical Wet-Day)"},
    {"th": 0.1, "status": "Sensitivity Candidate", "desc": "tau=0.1 mm (Y >= 0.1 mm, Light Rain Candidate)"}
]

for cfg in eval_configs:
    th = cfg["th"]
    wet_mask = (target >= th)
    n_wet = int(wet_mask.sum())
    pct_wet = (n_wet / n_total) * 100.0
    
    # Xác định các đợt mưa (precipitation episodes - contiguous wet blocks)
    # Mỗi đợt bắt đầu từ onset (Y_t >= th và Y_(t-1) < th)
    group_id = (wet_mask != wet_mask.shift(1, fill_value=False)).cumsum()
    wet_groups = group_id[wet_mask]
    episode_lengths = wet_groups.value_counts(sort=False)
    n_episodes = len(episode_lengths)
    
    # Boundary censoring inspection
    is_right_censored = bool(wet_mask.iloc[-1])
    n_right_censored = 1 if is_right_censored else 0
    mean_dur_all = float(episode_lengths.mean()) if n_episodes > 0 else 0.0
    
    if is_right_censored:
        last_group = wet_groups.iloc[-1]
        uncensored_lengths = episode_lengths.drop(last_group)
        mean_dur_uncensored = float(uncensored_lengths.mean()) if len(uncensored_lengths) > 0 else 0.0
    else:
        mean_dur_uncensored = mean_dur_all
    
    # Khoảng cách ngày lịch thực tế giữa các quan sát mưa (Calendar Mean Gap)
    wet_dates = dates[wet_mask].sort_values()
    intervals = wet_dates.diff().dt.days.dropna()
    calendar_mean_gap = float(intervals.mean())
    
    # ADI demand ratio (N / N_wet) theo định nghĩa taxonomy gián đoạn
    adi_ratio = float(n_total / n_wet)
    
    # CV và CV^2 tính trên quần thể lượng mưa dương của nhóm (Y >= th)
    wet_amounts = target[wet_mask]
    pos_m = float(wet_amounts.mean())
    pos_s = float(wet_amounts.std())
    cv_val = float(pos_s / pos_m) if pos_m > 0 else 0.0
    cv2_val = float(cv_val ** 2)
    
    # Phân loại Syntetos-Boylan (phép loại suy chẩn đoán từ chuỗi nhu cầu)
    if calendar_mean_gap < 1.32 and cv2_val < 0.49:
        sb_class = "SMOOTH-like"
    elif calendar_mean_gap < 1.32 and cv2_val >= 0.49:
        sb_class = "ERRATIC-like"
    elif calendar_mean_gap >= 1.32 and cv2_val < 0.49:
        sb_class = "INTERMITTENT-like"
    else:
        sb_class = "LUMPY-like (demand taxonomy analogy)"
    
    interm_records.append({
        "setting": cfg["desc"],
        "status": cfg["status"],
        "threshold_mm": th,
        "wet_days_count": n_wet,
        "wet_days_pct": round(pct_wet, 2),
        "n_episodes": n_episodes,
        "n_right_censored": n_right_censored,
        "mean_duration_all": round(mean_dur_all, 2),
        "mean_duration_uncensored": round(mean_dur_uncensored, 2),
        "calendar_mean_gap_days": round(calendar_mean_gap, 2),
        "adi_demand_ratio": round(adi_ratio, 2),
        "positive_mean_mm": round(pos_m, 2),
        "positive_cv_pct": round(cv_val * 100, 2),
        "cv_squared": round(cv2_val, 2),
        "syntetos_boylan_analogy": sb_class
    })

df_interm = pd.DataFrame(interm_records)
print("\n📊 [PHẦN 5] CHẨN ĐOÁN TÍNH GIÁN ĐOẠN (CANONICAL WET-DAY & EPISODE ROBUSTNESS):")
cols_show = ["setting", "wet_days_count", "wet_days_pct", "n_episodes", "n_right_censored", "mean_duration_all", "mean_duration_uncensored", "calendar_mean_gap_days", "adi_demand_ratio", "cv_squared"]
print(df_interm[cols_show].to_string(index=False))

row_10 = df_interm.loc[df_interm['threshold_mm'] == 1.0].iloc[0]
row_01 = df_interm.loc[df_interm['threshold_mm'] == 0.1].iloc[0]

print("\n💡 Ghi chú phương pháp luận quan trọng:")
print(f"   1. Canonical Wet-Day Threshold thống nhất: Y >= 1.0 mm (ETCCDI RR1 standard) đạt {row_10['wet_days_count']:,} ngày ({row_10['wet_days_pct']:.2f}%),")
print(f"      {row_10['n_episodes']:,} đợt mưa, thời lượng trung bình {row_10['mean_duration_all']:.2f} ngày/đợt. Thống nhất dùng xuyên suốt")
print("      cho EDA, Hurdle Stage 1 occurrence và rain-day diagnostics, không tách convention.")
print(f"   2. Boundary Censoring Robustness: Ở tau=0.1 mm, {row_01['n_right_censored']} đợt chịu right-censoring; mean duration all = {row_01['mean_duration_all']:.2f} vs uncensored = {row_01['mean_duration_uncensored']:.2f} ngày/đợt.")
print("   3. Tách biệt hai chỉ số: 'calendar_mean_gap_days' (khoảng cách ngày lịch thực tế) và 'adi_demand_ratio' (tỷ lệ N/N_wet).")
print(f"   4. Phép loại suy Syntetos-Boylan: Chuỗi rơi vào góc {row_10['syntetos_boylan_analogy']} cho thấy tính chất kết hợp giữa")
print("      sự gián đoạn xuất hiện và biên độ biến động mạnh, gợi ý thử nghiệm Two-Stage Hurdle và Tweedie.")



📊 [PHẦN 5] CHẨN ĐOÁN TÍNH GIÁN ĐOẠN (CANONICAL WET-DAY & EPISODE ROBUSTNESS):
                                       setting  wet_days_count  wet_days_pct  n_episodes  n_right_censored  mean_duration_all  mean_duration_uncensored  calendar_mean_gap_days  adi_demand_ratio  cv_squared
   tau=1.0 mm (Y >= 1.0 mm, Canonical Wet-Day)            2747         43.40         654                 1               4.20                      4.20                    2.30              2.30        1.25
tau=0.1 mm (Y >= 0.1 mm, Light Rain Candidate)            3857         60.94         466                 1               8.28                      8.28                    1.64              1.64        2.02

💡 Ghi chú phương pháp luận quan trọng:
   1. Canonical Wet-Day Threshold thống nhất: Y >= 1.0 mm (ETCCDI RR1 standard) đạt 2,747 ngày (43.40%),
      654 đợt mưa, thời lượng trung bình 4.20 ngày/đợt. Thống nhất dùng xuyên suốt
      cho EDA, Hurdle Stage 1 occurrence và rain-day diagnostics, không tác

In [4]:
# 7. Trực quan hóa Toàn diện Hình thái Phân phối & Tính Gián đoạn
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 7.1. Histogram Raw vs Log1p
ax1 = axes[0, 0]
ax1.hist(target, bins=50, color='#3498db', edgecolor='black', alpha=0.7, density=True, label=f'Raw Y (Skew={skew_all:.2f})')
ax1.axvline(mean_all, color='red', linestyle='--', linewidth=2, label=f'Mean ({mean_all:.2f} mm)')
ax1.axvline(median_all, color='green', linestyle='-', linewidth=2, label=f'Median ({median_all:.2f} mm)')
ax1.set_title('1. Phân phối Lượng mưa Thô Y_t (Lệch phải mạnh)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Lượng mưa (mm)')
ax1.set_ylabel('Mật độ (Density)')
ax1.set_xlim(-1, 60)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')

# 7.2. Log1p Transformed Distribution
ax2 = axes[0, 1]
ax2.hist(target_log1p, bins=40, color='#9b59b6', edgecolor='black', alpha=0.7, density=True, label=f'log(1+Y) (Skew={skew_log1p:.2f})')
ax2.axvline(target_log1p.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean ({target_log1p.mean():.2f})')
ax2.axvline(target_log1p.median(), color='green', linestyle='-', linewidth=2, label=f'Median ({target_log1p.median():.2f})')
ax2.set_title('2. Chẩn đoán Biến đổi log(1 + Y_t) (Giảm lệch đáng kể)', fontsize=11, fontweight='bold')
ax2.set_xlabel('log(1 + Lượng mưa)')
ax2.set_ylabel('Mật độ (Density)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')

# 7.3. Phân bố Cường độ Khí tượng Việt Nam
ax3 = axes[1, 0]
classes_labels = [row['Phân loại (Tiếng Việt)'].replace(' ', '\n') for row in vn_records]
classes_pcts = [row['Tỷ lệ (%)'] for row in vn_records]
bars = ax3.bar(classes_labels, classes_pcts, color='#e67e22', edgecolor='black', alpha=0.85)
for bar in bars:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2, h + 0.6, f'{h:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax3.set_title('3. Cường độ Mưa theo QCVN 47:2018/BTNMT & QĐ 18/2021/QĐ-TTg', fontsize=11, fontweight='bold')
ax3.set_xlabel('Cấp độ Mưa (Chuẩn Quốc gia)')
ax3.set_ylabel('Tỷ lệ phần trăm (%)')
ax3.set_ylim(0, 35)
ax3.grid(True, linestyle='--', alpha=0.5, axis='y')

# 7.4. Intermittency Taxonomy (Syntetos-Boylan Analogy Grid)
ax4 = axes[1, 1]
ax4.axvline(1.32, color='gray', linestyle='--', linewidth=1.5)
ax4.axhline(0.49, color='gray', linestyle='--', linewidth=1.5)
ax4.text(0.6, 0.25, 'SMOOTH', fontsize=11, ha='center', color='darkgreen', fontweight='bold')
ax4.text(0.6, 1.5, 'ERRATIC', fontsize=11, ha='center', color='darkorange', fontweight='bold')
ax4.text(2.0, 0.25, 'INTERMITTENT', fontsize=11, ha='center', color='darkblue', fontweight='bold')
ax4.text(2.0, 1.5, 'LUMPY', fontsize=11, ha='center', color='darkred', fontweight='bold')

# Plot candidate points
for rec in interm_records:
    lbl = rec['setting']
    is_canon = (rec['threshold_mm'] == 1.0)
    marker_style = '*' if is_canon else 'o'
    pt_size = 200 if is_canon else 130
    ax4.scatter(rec['calendar_mean_gap_days'], rec['cv_squared'], s=pt_size, marker=marker_style, zorder=5, label=f"{lbl} [Gap={rec['calendar_mean_gap_days']}d, CV2={rec['cv_squared']}]")
    ax4.annotate(lbl.split(' ')[0], (rec['calendar_mean_gap_days'] + 0.05, rec['cv_squared'] + 0.05), fontsize=8, fontweight='bold')

ax4.set_title('4. Phép Loại suy Tính Gián đoạn (Syntetos-Boylan Analogy)', fontsize=11, fontweight='bold')
ax4.set_xlabel('Khoảng cách ngày lịch trung bình giữa các lần mưa [Ngưỡng 1.32]')
ax4.set_ylabel('CV² Lượng mưa Dương [Ngưỡng 0.49]')
ax4.set_xlim(0.0, 3.0)
ax4.set_ylim(0.0, 2.5)
ax4.grid(True, linestyle='--', alpha=0.5)
ax4.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()


In [5]:
# 8. Xuất Artifact Báo cáo Phân phối và Tính Gián đoạn (Phân định Rõ Bằng chứng vs Khuyến nghị)
OUTPUT_DIR = ROOT_DIR / "reports" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

row_10 = df_interm.loc[df_interm['threshold_mm'] == 1.0].iloc[0]
row_01 = df_interm.loc[df_interm['threshold_mm'] == 0.1].iloc[0]
last_date = train_df['Ngày'].max().date()
last_val = float(target.iloc[-1])

dist_artifact = {
    "target_variable": "Lượng mưa",
    "metadata": {
        "data_split": "dev_train",
        "train_start": str(train_df['Ngày'].min().date()),
        "train_end": str(last_date),
        "n_observations": n_total,
        "holdouts_untouched": "Validation (2018-05-01 to 2020-04-30, N=731) and Final Test (2020-05-01 to 2026-04-30, N=2191) strictly untouched."
    },
    "sample_size": n_total,
    "distribution_analysis_threshold_independent": {
        "mean_mm": round(mean_all, 4),
        "median_mm": round(median_all, 4),
        "mode_mm": 0.0,
        "std_mm": round(std_all, 4),
        "overall_cv_pct": round(cv_all, 2),
        "skewness": round(skew_all, 4),
        "excess_kurtosis": round(kurt_all, 4),
        "q1_mm": round(q1_all, 4),
        "q3_mm": round(q3_all, 4),
        "iqr_mm": round(iqr_all, 4),
        "p95_mm": round(p95_all, 4),
        "p99_mm": round(p99_all, 4),
        "max_mm": round(max_all, 4)
    },
    "zero_positive_decomposition": {
        "exact_zero_count": n_zero,
        "exact_zero_pct": round(pct_zero, 2),
        "positive_rainfall_count": n_pos,
        "positive_rainfall_pct": round(pct_pos, 2),
        "positive_mean_mm": round(pos_mean, 4),
        "positive_std_mm": round(pos_std, 4),
        "positive_median_mm": round(pos_median, 4),
        "positive_cv_pct": round(pos_cv, 2)
    },
    "shape_transformation_diagnostics": {
        "raw_skewness": round(skew_all, 4),
        "raw_kurtosis": round(kurt_all, 4),
        "log1p_skewness": round(skew_log1p, 4),
        "log1p_kurtosis": round(kurt_log1p, 4),
        "finding": f"log1p transformation substantially reduces skewness from {skew_all:.2f} to {skew_log1p:.2f} and brings kurtosis to {kurt_log1p:.2f}, indicating that a transformed target representation is a promising candidate to evaluate across suitable model families."
    },
    "vietnamese_meteorological_classes": df_vn_classes.to_dict(orient="records"),
    "vietnamese_standard_reference": (
        "QCVN 47:2018/BTNMT (Quy chuẩn kỹ thuật quốc gia về dự báo, cảnh báo khí tượng thủy văn) "
        "& Quyết định số 18/2021/QĐ-TTg của Thủ tướng Chính phủ."
    ),
    "intermittency_diagnostics_candidate_dependent": {
        "canonical_wet_day_threshold": {
            "threshold_mm": 1.0,
            "condition": "Y >= 1.0 mm",
            "wet_days_count": int(row_10['wet_days_count']),
            "wet_days_pct": float(row_10['wet_days_pct']),
            "n_episodes": int(row_10['n_episodes']),
            "mean_duration_all": float(row_10['mean_duration_all']),
            "mean_duration_uncensored": float(row_10['mean_duration_uncensored']),
            "calendar_mean_gap_days": float(row_10['calendar_mean_gap_days']),
            "adi_demand_ratio": float(row_10['adi_demand_ratio']),
            "positive_mean_mm": float(row_10['positive_mean_mm']),
            "cv_squared": float(row_10['cv_squared']),
            "climate_reference": "Adopted from ETCCDI RR1 standard (precipitation >= 1.0 mm/day defines a wet day), unified across EDA and Hurdle Stage 1."
        },
        "candidate_evaluations": df_interm.to_dict(orient="records"),
        "episode_boundary_policy": {
            "analysis_window": f"2001-01-01 to {last_date} ({n_total:,} days)",
            "left_censored": False,
            "left_censoring_note": "First observation 2001-01-01 has 0.0 mm rainfall (dry day), so no initial episode is left-censored.",
            "right_censored_tau_0_1": bool(row_01['n_right_censored'] > 0),
            "right_censored_tau_1_0": bool(row_10['n_right_censored'] > 0),
            "right_censoring_note": f"Final observation {last_date} has {last_val:.2f} mm rainfall. Right-censored status: tau=0.1: {bool(row_01['n_right_censored'] > 0)}; tau=1.0: {bool(row_10['n_right_censored'] > 0)}.",
            "policy_statement": "Observed episode duration statistics within the analysis window are reported alongside uncensored mean duration to verify boundary robustness."
        },
        "methodology_notes": {
            "adi_nature": "adi_demand_ratio (N / N_wet) directly reflects inverse wet-day frequency, distinct from calendar_mean_gap_days.",
            "episode_definition": "Precipitation episodes are defined empirically by threshold onsets (Y_t >= tau and Y_(t-1) < tau), representing threshold-defined episodes rather than physical storm systems.",
            "syntetos_boylan_status": "Exploratory analogy transferred from intermittent demand forecasting literature; not an official meteorological standard."
        }
    },
    "observed_evidence": [
        f"Target distribution Y_t is strongly right-skewed (skewness = {skew_all:.2f}) with a heavy leptokurtic tail (excess kurtosis = {kurt_all:.2f}, P95 = {p95_all:.2f} mm, Max = {max_all:.2f} mm).",
        f"Target exhibits a substantial exact-zero mass ({pct_zero:.2f}% of observations).",
        f"Positive rainfall population remains highly variable (positive CV = {pos_cv:.2f}%, overall CV = {cv_all:.2f}%).",
        f"log1p transformation substantially reduces skewness (from {skew_all:.2f} to {skew_log1p:.2f}) and tail heaviness.",
        f"Unified canonical wet-day definition (Y >= 1.0 mm, ETCCDI RR1) defines {int(row_10['wet_days_count']):,} wet days ({float(row_10['wet_days_pct']):.2f}%) across {int(row_10['n_episodes']):,} episodes (mean duration {float(row_10['mean_duration_all']):.2f} days).",
        "Higher threshold tau (0.1 -> 1.0 mm) reduces wet days and segments long continuous low-intensity periods into shorter, more separated wet episodes."
    ],
    "modeling_implications": [
        "The empirical distribution indicates that a simple symmetric Gaussian assumption does not adequately reflect the marginal structure of rainfall.",
        "Motivates evaluation of Two-Stage Hurdle models (separating occurrence Y >= 1.0 mm from conditional amount Y >= 1.0 mm).",
        "Motivates evaluation of Tweedie Compound Poisson GLM (capable of handling zero mass and continuous positive amounts).",
        "Motivates evaluating transformed target representations (such as log1p) against raw targets on validation splits, requiring expm1 inverse-transformation before evaluating continuous forecast metrics.",
        "Motivates non-linear machine learning and temporal models capable of capturing intermittent bursts."
    ],
    "not_conclusions_from_eda": [
        "EDA does NOT prove standard OLS regression will fail (OLS can still provide useful point forecasts even when the marginal target is non-Gaussian).",
        "EDA does NOT establish that Tweedie or Hurdle is guaranteed to be superior to other model families.",
        "EDA does NOT require that models must train on log1p targets.",
        "EDA does NOT prove that tau = 1.0 mm optimizes downstream forecasting accuracy.",
        "All model selections and performance conclusions are strictly deferred to downstream validation-set benchmarking."
    ]
}

artifact_path = OUTPUT_DIR / "01.2_distribution_report.json"
with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(dist_artifact, f, indent=2, ensure_ascii=False)

print(f"\n💾 Distribution & intermittency report exported to: {artifact_path}")
print("   • Target distribution: Evaluated continuously on Y_t >= 0 (threshold-independent)")
print("   • Unified canonical wet-day threshold: Y >= 1.0 mm across EDA and Hurdle")
print("   • Episode duration robustness: Reported both mean_duration_all and mean_duration_uncensored")
print("   • Architectural discipline: Clean separation among observed_evidence, modeling_implications, and not_conclusions_from_eda")



💾 Distribution & intermittency report exported to: D:\DS108_project\reports\eda\01.2_distribution_report.json
   • Target distribution: Evaluated continuously on Y_t >= 0 (threshold-independent)
   • Unified canonical wet-day threshold: Y >= 1.0 mm across EDA and Hurdle
   • Episode duration robustness: Reported both mean_duration_all and mean_duration_uncensored
   • Architectural discipline: Clean separation among observed_evidence, modeling_implications, and not_conclusions_from_eda
